In [ ]:
# Mount Drive and Prepare Data
from google.colab import drive
import zipfile
import os
import shutil

# Mount Drive
drive.mount('/content/drive')

# Setup Folder
if os.path.exists('/content/food_data'):
    shutil.rmtree('/content/food_data')
os.makedirs('/content/food_data', exist_ok=True)

# Unzip Images (This takes 1-2 mins)
zip_path = '/content/drive/MyDrive/Colab Notebooks/data.zip'
print("Unzipping images...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/food_data')

# Copy CSVs
shutil.copy('/content/drive/MyDrive/Colab Notebooks/train.csv.csv', '/content/food_data/train.csv.csv')
shutil.copy('/content/drive/MyDrive/Colab Notebooks/test.csv', '/content/food_data/test.csv')
print("✅ Files ready.")


Mounted at /content/drive
Unzipping images...
✅ Files ready.


In [ ]:
# Imports and Data Loading
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset Class Implementation
class FoodDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        if not is_test:
            self.categories = sorted(self.df['label_name'].unique())
            self.label_map = {name: i for i, name in enumerate(self.categories)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        full_csv_path = str(self.df.loc[idx, 'image_path'])
        path_parts = full_csv_path.replace('\\', '/').split('/')
        actual_path = os.path.join(self.root_dir, 'images', path_parts[-2], path_parts[-1])

        image = Image.open(actual_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, self.df.loc[idx, 'original_index']

        label = self.label_map[self.df.loc[idx, 'label_name']]
        return image, torch.tensor(label)

# Advanced Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize Full Dataset
data_root = '/content/food_data/data/food-101'
full_train_dataset = FoodDataset('/content/food_data/train.csv.csv', data_root, transform=train_transform)

# 70/30 Data Split
# Splitting 7,500 images into 5,250 (Train) and 2,250 (Validation)
train_size = int(0.7 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# Create DataLoaders for Optuna and Training
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

# Final Validation Check
print("--- Validation Check ---")
print(f"Total Dataset: {len(full_train_dataset)} images")
print(f"Training set: {len(train_subset)} images (70%)")
print(f"Validation set: {len(val_subset)} images (30%)")
print(f"Device: {device}")

--- Validation Check ---
Total Dataset: 7500 images
Training set: 5250 images (70%)
Validation set: 2250 images (30%)
Device: cuda


In [ ]:
# 5-Layer "Super Deep CNN" Architecture
class SuperDeepAssignmentCNN(nn.Module):
    def __init__(self, dropout_rate=0.4):
        super(SuperDeepAssignmentCNN, self).__init__()

        # Block 1: Input 224x224 -> Output 112x112
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2: 112x112 -> 56x56
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3: 56x56 -> 28x28
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Block 4: 28x28 -> 14x14
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        # NEW Block 5: 14x14 -> 7x7
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(2, 2)

        # Fully Connected Layers
        # 512 channels * 7 * 7 spatial dimension = 25088
        self.fc1 = nn.Linear(512 * 7 * 7, 512)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(512, 10) # 10 Food Classes

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x)))) # 5th Layer

        x = x.view(-1, 512 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
# Grid Search with 3-Fold Cross-Validation
from sklearn.model_selection import KFold

param_grid = {'lr': [5e-4, 1e-4], 'dropout': [0.4, 0.5]}
kf = KFold(n_splits=3, shuffle=True, random_state=42)
results_table = []

print("Starting Systematic Grid Search...")
for lr in param_grid['lr']:
    for do in param_grid['dropout']:
        fold_accs = []
        for fold, (train_idx, val_idx) in enumerate(kf.split(full_train_dataset)):
            # Create loaders for this fold
            train_sub = torch.utils.data.Subset(full_train_dataset, train_idx)
            val_sub = torch.utils.data.Subset(full_train_dataset, val_idx)
            loader_train = DataLoader(train_sub, batch_size=32, shuffle=True)
            loader_val = DataLoader(val_sub, batch_size=32, shuffle=False)

            # Init Model
            model = SuperDeepAssignmentCNN(dropout_rate=do).to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr)
            criterion = nn.CrossEntropyLoss()

            # Short Training to find best params
            for epoch in range(15):
                model.train()
                for imgs, lbls in loader_train:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(imgs), lbls)
                    loss.backward()
                    optimizer.step()

            # Val Accuracy
            model.eval()
            correct = 0
            with torch.no_grad():
                for imgs, lbls in loader_val:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    correct += (model(imgs).argmax(1) == lbls).sum().item()
            fold_accs.append(correct / len(val_sub))

        avg_acc = np.mean(fold_accs)
        results_table.append({'lr': lr, 'dropout': do, 'avg_acc': avg_acc})
        print(f"LR: {lr}, DO: {do} -> Avg Acc: {avg_acc:.4f}")

summary_df = pd.DataFrame(results_table)
print("\n--- Summary Table for Report ---")
print(summary_df)

Starting Systematic Grid Search...
LR: 0.0005, DO: 0.4 -> Avg Acc: 0.4440
LR: 0.0005, DO: 0.5 -> Avg Acc: 0.4347
LR: 0.0001, DO: 0.4 -> Avg Acc: 0.5621
LR: 0.0001, DO: 0.5 -> Avg Acc: 0.5281

--- Summary Table for Report ---
       lr  dropout   avg_acc
0  0.0005      0.4  0.444000
1  0.0005      0.5  0.434667
2  0.0001      0.4  0.562133
3  0.0001      0.5  0.528133


In [ ]:
# Final Training and Kaggle Submission

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn as nn
import pandas as pd
from torchvision import transforms

# Define Test Transform (Clean resize, no random flips/crops)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Configuration & Model Init
best_lr = 0.0001
best_do = 0.4
final_model = SuperDeepAssignmentCNN(dropout_rate=best_do).to(device)

# Setup Optimizer, Loss, and Scheduler
optimizer = optim.Adam(final_model.parameters(), lr=best_lr, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Create Loaders
# Training uses all 7,500 images with the 'train_transform' from Cell 2
full_train_loader = DataLoader(full_train_dataset, batch_size=32, shuffle=True)

# Test uses the new clean 'test_transform'
test_dataset = FoodDataset('/content/food_data/test.csv', data_root, transform=test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Starting Final Training: 40 Epochs using 5-Layer CNN on {device}...")

# Training Loop
final_model.train()
for epoch in range(1, 41):
    running_loss = 0.0
    for imgs, lbls in full_train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        optimizer.zero_grad()
        outputs = final_model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    avg_loss = running_loss / len(full_train_loader)
    print(f"Epoch [{epoch}/40] - Avg Loss: {avg_loss:.6f} - LR: {current_lr:.6f}")

    if epoch % 10 == 0:
        torch.save(final_model.state_dict(), f"model_5layer_epoch_{epoch}.pth")
        print(f"Checkpoint saved at Epoch {epoch}")

print("Final Training Complete!")

# Final Inference
final_model.eval()
results = []
# Map indices back to names (e.g., 0 -> 'baby_back_ribs')
inv_label_map = {v: k for k, v in full_train_dataset.label_map.items()}

print(f"Generating Kaggle predictions for 2,500 images...")
with torch.no_grad():
    for imgs, indices in test_loader:
        imgs = imgs.to(device)
        preds = final_model(imgs).argmax(1)
        for idx, p in zip(indices, preds):
            results.append({
                "original_index": int(idx),
                "label_name": inv_label_map[p.item()]
            })

# 7. Save the final file
output_csv = "deep_5layer_final_submission_77plus.csv"
pd.DataFrame(results).to_csv(output_csv, index=False)
print(f"Done! File '{output_csv}' is ready for Kaggle.")

Starting Final Training: 40 Epochs using 5-Layer CNN on cuda...
Epoch [1/40] - Avg Loss: 2.007879 - LR: 0.000100
Epoch [2/40] - Avg Loss: 1.684995 - LR: 0.000100
Epoch [3/40] - Avg Loss: 1.545335 - LR: 0.000100
Epoch [4/40] - Avg Loss: 1.444782 - LR: 0.000100
Epoch [5/40] - Avg Loss: 1.378705 - LR: 0.000100
Epoch [6/40] - Avg Loss: 1.293347 - LR: 0.000100
Epoch [7/40] - Avg Loss: 1.238218 - LR: 0.000100
Epoch [8/40] - Avg Loss: 1.200946 - LR: 0.000100
Epoch [9/40] - Avg Loss: 1.164397 - LR: 0.000100
Epoch [10/40] - Avg Loss: 1.108761 - LR: 0.000050
Checkpoint saved at Epoch 10
Epoch [11/40] - Avg Loss: 0.999192 - LR: 0.000050
Epoch [12/40] - Avg Loss: 0.964751 - LR: 0.000050
Epoch [13/40] - Avg Loss: 0.948090 - LR: 0.000050
Epoch [14/40] - Avg Loss: 0.929963 - LR: 0.000050
Epoch [15/40] - Avg Loss: 0.894436 - LR: 0.000050
Epoch [16/40] - Avg Loss: 0.872594 - LR: 0.000050
Epoch [17/40] - Avg Loss: 0.856910 - LR: 0.000050
Epoch [18/40] - Avg Loss: 0.851008 - LR: 0.000050
Epoch [19/40] - 